# LangGraph MCP - ArXiv 튜토리얼

이 튜토리얼에서는 **ArXiv MCP** 서버를 LangGraph와 통합하여 arXiv 논문을 검색하고, 상세 정보를 조회하며, 요약 분석하는 AI 에이전트를 구축하는 방법을 학습합니다.

ArXiv MCP 서버는 세계 최대의 오픈 액세스 학술 논문 저장소인 [arXiv.org](https://arxiv.org/)에 MCP 프로토콜을 통해 접근할 수 있게 해주는 서버입니다. API 키 없이 공개된 arXiv 데이터에 접근할 수 있으며, 논문 검색, 메타데이터 조회, PDF 내용 추출 등의 기능을 제공합니다.

> 참고 문서: [ArXiv MCP GitHub](https://github.com/kelvingao/arxiv-mcp) | [arXiv.org](https://arxiv.org/)

## 학습 목표

1. ArXiv MCP 서버의 개념과 제공 도구를 이해합니다
2. ArXiv MCP 서버를 설치하고 stdio 방식으로 연결하는 방법을 학습합니다
3. `search_arxiv`, `get_paper_details`, `search_and_summarize` 도구의 사용법을 익힙니다
4. `create_agent`를 사용하여 ArXiv 논문 검색 AI 에이전트를 구축합니다
5. ToolNode를 활용한 커스텀 워크플로우와 ArXiv MCP를 통합합니다
6. 실전 예제를 통해 학술 논문 검색 및 분석을 경험합니다

## 목차

1. ArXiv MCP 개요 및 설치
2. 환경 설정
3. ArXiv MCP 연결 기본
4. 도구 직접 호출
5. Agent와 ArXiv MCP 통합
6. 대화 컨텍스트 유지 (다중 턴 대화)
7. ToolNode 기반 커스텀 워크플로우
8. 실전 활용 예제

## 환경 설정

튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 과정을 확인할 수 있도록 합니다.

LangSmith 추적을 활성화하면 에이전트의 추론 과정, 도구 호출, 응답 생성 등을 시각적으로 모니터링할 수 있어 디버깅에 큰 도움이 됩니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)

# LangSmith 추적 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

---

## Part 1: ArXiv MCP 개요 및 설치

### ArXiv MCP란?

ArXiv MCP 서버는 세계 최대의 오픈 액세스 학술 논문 저장소인 [arXiv.org](https://arxiv.org/)에 MCP 프로토콜을 통해 접근할 수 있게 해주는 서버입니다. 물리학, 수학, 컴퓨터 과학, 생물학 등 다양한 분야의 논문을 검색하고 분석할 수 있습니다.

### ArXiv MCP가 해결하는 문제

| 기존 문제 | ArXiv MCP 해결책 |
|-----------|------------------|
| arXiv 웹사이트 수동 검색 필요 | MCP 도구를 통한 자동화된 논문 검색 |
| 논문 PDF 직접 다운로드 및 읽기 | PDF 내용 자동 추출 및 텍스트 변환 |
| 여러 논문 비교 분석에 시간 소요 | LLM 기반 자동 요약 및 비교 분석 |
| API 키 발급 및 관리 필요 | API 키 없이 공개 arXiv 데이터 접근 |

### 제공 도구

ArXiv MCP 서버는 세 가지 MCP 도구를 제공합니다:

#### 1. `search_arxiv`
키워드로 arXiv 논문을 검색합니다.
- **입력**: `query` (검색 키워드), `limit` (결과 수 제한, 기본값 5)
- **출력**: 검색된 논문 목록 (제목, 저자, 요약 등)

#### 2. `get_paper_details`
특정 논문의 상세 정보를 조회합니다.
- **입력**: `paper_id` (arXiv 논문 ID, 예: "2301.00001"), `include_content` (PDF 내용 포함 여부, 기본값: False)
- **출력**: 논문의 상세 메타데이터 및 선택적으로 PDF 전문

#### 3. `search_and_summarize`
논문을 검색하고 종합적인 요약을 제공합니다.
- **입력**: `query` (검색 키워드), `limit` (결과 수 제한)
- **출력**: 검색된 논문들의 마크다운 형식 종합 요약

### 설치 방법

ArXiv MCP 서버는 GitHub에서 클론하여 설치합니다. Python 3.11 이상이 필요합니다.

```bash
# 1. 저장소 클론
git clone https://github.com/kelvingao/arxiv-mcp.git

# 2. 디렉토리 이동
cd arxiv-mcp

# 3. 패키지 설치 (uv 사용)
uv pip install -e .

# 4. 환경 변수 설정
# .env.example을 복사하여 .env 생성 후 TRANSPORT=stdio 설정
cp .env.example .env
# .env 파일에서 TRANSPORT=stdio 로 설정
```

> **참고**: ArXiv MCP는 공개 arXiv 데이터를 사용하므로 별도의 API 키가 필요하지 않습니다.

---

## Part 2: 기본 설정 및 패키지 임포트

ArXiv MCP 서버를 사용하기 위해 필요한 패키지들을 임포트합니다. `langchain-mcp-adapters`는 MCP 서버를 LangChain 컴포넌트와 연결하는 핵심 라이브러리입니다.

```bash
# 필요한 패키지 설치 (이미 설치되어 있다면 생략)
uv add langchain-mcp-adapters
```

In [ ]:
import nest_asyncio
from typing import List, Dict, Any

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트: 여러 MCP 서버에 연결하여 도구를 가져옵니다
from langchain_mcp_adapters.client import MultiServerMCPClient

# 비동기 호환을 활성화합니다 (Jupyter 환경에서 필요)
nest_asyncio.apply()

---

## Part 3: ArXiv MCP 연결 기본

### MCP 클라이언트 설정 헬퍼 함수

`MultiServerMCPClient`를 사용하면 여러 MCP 서버에 동시에 연결하고 각 서버의 도구를 통합하여 사용할 수 있습니다. 아래는 Windows 환경과 Jupyter 호환성 패치와 함께 MCP 클라이언트를 설정하는 헬퍼 함수입니다.

**Windows 호환성 패치**: Windows 환경에서 Jupyter Notebook을 사용할 때 MCP stdio가 Jupyter의 `sys.stderr`를 서브프로세스에 전달하면서 `fileno()` 에러가 발생할 수 있습니다. 이를 방지하기 위한 패치를 적용합니다.

In [ ]:
import sys, os

# Windows + Jupyter workaround: MCP stdio passes Jupyter's sys.stderr to subprocess.Popen,
# but Jupyter's stderr doesn't support fileno(). Patch the default errlog to os.devnull.
if sys.platform == "win32":
    import mcp.client.stdio as _mcp_stdio

    _devnull_file = open(os.devnull, "w")

    # @asynccontextmanager wraps the original function — patch __wrapped__.__defaults__
    if hasattr(_mcp_stdio.stdio_client, "__wrapped__"):
        _mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_devnull_file,)

    # Also patch the helper that creates the subprocess
    _mcp_stdio._create_platform_compatible_process.__defaults__ = (
        None,
        _devnull_file,
        None,
    )


async def setup_mcp_client(server_configs: dict):
    """MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 구성 딕셔너리. 각 서버의 이름을 키로,
                       연결 정보(command, args, transport 또는 url)를 값으로 가집니다.

    Returns:
        tuple: (MCP 클라이언트, 로드된 도구 목록)
    """
    # MCP 클라이언트 생성
    client = MultiServerMCPClient(server_configs)

    # 서버에 연결하여 도구 목록을 가져옵니다
    tools = await client.get_tools()

    # 로드된 도구 목록을 출력합니다
    print(f"[MCP] {len(tools)}개의 도구가 로드되었습니다:")
    for tool in tools:
        print(
            f"  - {tool.name}: {tool.description[:80]}..."
            if len(tool.description) > 80
            else f"  - {tool.name}: {tool.description}"
        )

    return client, tools

### ArXiv MCP 서버 연결

ArXiv MCP 서버는 `stdio` 전송 방식으로 연결합니다. 사전에 저장소를 클론하고 패키지를 설치해야 합니다. `TRANSPORT` 환경 변수를 `stdio`로 설정하면 표준 입출력을 통해 통신합니다.

**서버 구성 설명:**
- `command`: 클론한 저장소의 가상환경 Python 실행 경로
- `args`: `["src/server.py"]` — ArXiv MCP 서버 스크립트
- `transport`: `stdio` — 표준 입출력을 통한 통신
- `env`: `TRANSPORT=stdio` — 서버가 stdio 모드로 동작하도록 설정

> **경로 설정**: 아래 `ARXIV_MCP_PATH` 변수를 실제 클론한 경로로 변경하세요.

In [ ]:
# ArXiv MCP 서버가 클론된 경로를 설정하세요
# 예: git clone https://github.com/kelvingao/arxiv-mcp.git 으로 클론한 위치
ARXIV_MCP_PATH = os.path.expanduser("~/arxiv-mcp")

# ArXiv MCP 서버 구성 (stdio 전송 방식)
arxiv_server_config = {
    "arxiv": {
        "command": os.path.join(
            ARXIV_MCP_PATH,
            ".venv",
            "Scripts" if sys.platform == "win32" else "bin",
            "python",
        ),
        "args": [os.path.join(ARXIV_MCP_PATH, "src", "server.py")],
        "transport": "stdio",
        "env": {
            "TRANSPORT": "stdio",
        },
    }
}

# MCP 클라이언트 생성 및 도구 로드
client, tools = await setup_mcp_client(server_configs=arxiv_server_config)

### 도구 상세 정보 확인

로드된 ArXiv MCP 도구의 이름, 설명, 파라미터 스키마를 상세히 확인합니다.

In [ ]:
import json

# 로드된 도구 상세 정보 출력
for i, tool in enumerate(tools):
    print(f"\n{'='*60}")
    print(f"도구 {i+1}: {tool.name}")
    print(f"{'='*60}")
    print(f"설명: {tool.description}")
    print(f"\n파라미터 스키마:")
    if hasattr(tool, "args_schema") and tool.args_schema:
        schema = (
            tool.args_schema.schema()
            if hasattr(tool.args_schema, "schema")
            else tool.args_schema
        )
        print(json.dumps(schema, indent=2, ensure_ascii=False))

---

## Part 4: 도구 직접 호출

### search_arxiv 도구 직접 호출

`search_arxiv`는 키워드 기반으로 arXiv 논문을 검색합니다. 에이전트를 거치지 않고 도구를 직접 호출하여 결과를 확인합니다.

| 파라미터 | 필수 | 기본값 | 설명 |
|----------|------|--------|------|
| `query` | O | - | 검색 키워드 또는 문구 |
| `limit` | X | 5 | 반환할 최대 결과 수 |

In [ ]:
# search_arxiv 도구 가져오기
search_tool = next(t for t in tools if t.name == "search_arxiv")

# "transformer architecture" 키워드로 논문 검색 (최대 3건)
result = await search_tool.ainvoke({
    "query": "transformer architecture attention mechanism",
    "limit": 3,
})

print("arXiv 논문 검색 결과:")
print("-" * 60)
print(result[:3000] if len(str(result)) > 3000 else result)

### get_paper_details 도구 직접 호출

`get_paper_details`는 arXiv 논문 ID를 사용하여 특정 논문의 상세 정보를 조회합니다. `include_content=True`로 설정하면 PDF에서 전문을 추출합니다.

| 파라미터 | 필수 | 기본값 | 설명 |
|----------|------|--------|------|
| `paper_id` | O | - | arXiv 논문 ID (예: "1706.03762") |
| `include_content` | X | False | PDF 내용 추출 여부 |

> **참고**: 유명한 논문 ID 예시 — "Attention Is All You Need" 논문: `1706.03762`

In [ ]:
# get_paper_details 도구 가져오기
details_tool = next(t for t in tools if t.name == "get_paper_details")

# "Attention Is All You Need" 논문 상세 정보 조회
result = await details_tool.ainvoke({
    "paper_id": "1706.03762",
    "include_content": False,
})

print("논문 상세 정보 (Attention Is All You Need):")
print("-" * 60)
print(result[:3000] if len(str(result)) > 3000 else result)

### search_and_summarize 도구 직접 호출

`search_and_summarize`는 논문 검색과 종합 요약을 한 번에 수행합니다. 검색된 논문들의 메타데이터와 내용을 마크다운 형식으로 정리하여 반환합니다.

| 파라미터 | 필수 | 기본값 | 설명 |
|----------|------|--------|------|
| `query` | O | - | 검색 키워드 또는 문구 |
| `limit` | X | 5 | 반환할 최대 결과 수 |

In [ ]:
# search_and_summarize 도구 가져오기
summarize_tool = next(t for t in tools if t.name == "search_and_summarize")

# "large language model alignment" 키워드로 논문 검색 및 요약
result = await summarize_tool.ainvoke({
    "query": "large language model alignment RLHF",
    "limit": 3,
})

print("논문 검색 및 요약 결과:")
print("-" * 60)
print(result[:5000] if len(str(result)) > 5000 else result)

---

## Part 5: Agent와 ArXiv MCP 통합

### create_agent 기반 ArXiv 에이전트

`create_agent`는 LLM과 도구 목록을 전달하면 추론-행동(Reason-Act) 루프를 자동으로 구현하는 에이전트를 생성합니다. ArXiv 도구를 에이전트에 통합하면, LLM이 사용자의 질문에 따라 자동으로 적절한 도구(`search_arxiv`, `get_paper_details`, `search_and_summarize`)를 선택하여 호출합니다.

> 참고: LangGraph v1에서 기존의 `create_react_agent`는 deprecated 되었으며, `langchain.agents.create_agent`를 사용하는 것이 권장됩니다.

아래 코드는 ArXiv 도구를 사용하는 에이전트를 생성하는 헬퍼 함수를 정의합니다.

In [ ]:
async def create_arxiv_agent():
    """ArXiv MCP 서버를 사용하는 에이전트를 생성합니다.

    Returns:
        CompiledStateGraph: 컴파일된 에이전트
    """
    # ArXiv MCP 서버 구성
    server_configs = {
        "arxiv": {
            "command": os.path.join(
                ARXIV_MCP_PATH,
                ".venv",
                "Scripts" if sys.platform == "win32" else "bin",
                "python",
            ),
            "args": [os.path.join(ARXIV_MCP_PATH, "src", "server.py")],
            "transport": "stdio",
            "env": {
                "TRANSPORT": "stdio",
            },
        }
    }

    # MCP 클라이언트 생성 및 도구 로드
    client, tools = await setup_mcp_client(server_configs=server_configs)

    # LLM 설정
    llm = init_chat_model("claude-sonnet-4-6", temperature=0)

    # 에이전트 생성: ArXiv 도구를 사용하는 에이전트
    agent = create_agent(
        llm,
        tools,
        checkpointer=InMemorySaver(),  # 대화 상태를 메모리에 저장
    )

    return agent

In [ ]:
# ArXiv 에이전트 생성
agent = await create_arxiv_agent()

In [ ]:
# 에이전트 그래프 구조 확인
agent

### 최신 논문 검색

에이전트가 ArXiv 도구를 사용하여 특정 주제의 최신 논문을 검색하고 결과를 정리합니다. 에이전트는 사용자의 질문을 분석하여 자동으로 적절한 도구를 선택합니다.

In [ ]:
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

# 대화 스레드 ID 설정
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 최신 RAG 관련 논문 검색 요청
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "2024년 이후 발표된 RAG(Retrieval-Augmented Generation) 관련 최신 논문을 3편 검색해주세요. "
                "각 논문의 제목, 저자, 핵심 기여를 정리해주세요.",
            )
        ]
    },
    config=config,
)

### 특정 논문 상세 분석

arXiv 논문 ID를 지정하여 특정 논문의 상세 정보를 조회하고 분석합니다.

In [ ]:
# 새로운 대화 스레드 (독립적인 대화 시작)
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 특정 논문 상세 분석 요청
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "arXiv 논문 ID '2005.11401'의 상세 정보를 조회하고, "
                "이 논문의 핵심 방법론과 기여를 설명해주세요.",
            )
        ]
    },
    config=config,
)

### 분야별 논문 요약

특정 연구 분야의 논문을 검색하고 종합적으로 요약합니다. 에이전트가 `search_and_summarize` 도구를 활용하여 마크다운 형식의 체계적인 요약을 제공합니다.

In [ ]:
# 새로운 대화 스레드
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 멀티모달 LLM 관련 논문 검색 및 요약
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "멀티모달 대규모 언어 모델(Multimodal LLM)에 대한 최신 논문을 검색하고 "
                "종합적으로 요약해주세요.",
            )
        ]
    },
    config=config,
)

---

## Part 6: 대화 컨텍스트 유지 (다중 턴 대화)

동일한 `thread_id`를 사용하면 이전 대화 내용이 유지됩니다. 논문 검색 후 후속 질문을 하면 에이전트가 이전 검색 결과를 참조하여 답변합니다. 이 기능은 논문 리뷰 과정에서 매우 유용합니다.

In [ ]:
# 같은 thread_id로 연속 대화 (컨텍스트 유지)
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 첫 번째 질문: 논문 검색
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "Graph Neural Network 관련 최신 논문을 2편 검색해주세요.",
            )
        ]
    },
    config=config,
)

In [ ]:
# 두 번째 질문: 이전 대화 컨텍스트를 유지하면서 후속 질문
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "위에서 검색한 논문들 중 첫 번째 논문의 상세 정보를 조회하고, "
                "이 논문이 기존 GNN 연구와 어떻게 다른지 분석해주세요.",
            )
        ]
    },
    config=config,
)

---

## Part 7: ToolNode 기반 커스텀 워크플로우

### ToolNode와 ArXiv MCP

`ToolNode`를 사용하면 LangGraph에서 더 세밀한 제어가 가능한 커스텀 워크플로우를 만들 수 있습니다. `create_agent`와 달리, 그래프의 각 노드를 직접 정의하고 연결하여 복잡한 로직을 구현할 수 있습니다.

ArXiv MCP와 Tavily 검색 도구를 결합하면 **학술 논문 검색 + 웹 검색**을 동시에 활용하는 강력한 에이전트를 만들 수 있습니다.

### 워크플로우 구조

```
START → [agent 노드] → tools_condition → [tools 노드] → [agent 노드] → ...
                                        ↘ END (도구 호출 없을 때)
```

아래 코드는 ArXiv MCP + Tavily 검색을 결합한 커스텀 워크플로우를 정의합니다.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from typing import Annotated, TypedDict
from langchain_tavily import TavilySearch


class AgentState(TypedDict):
    """에이전트 상태 정의

    Attributes:
        messages: 대화 메시지 목록. add_messages 리듀서로 메시지가 누적됩니다.
    """

    messages: Annotated[List[BaseMessage], add_messages]


async def create_arxiv_workflow():
    """ArXiv MCP + Tavily 검색을 결합한 커스텀 워크플로우를 생성합니다.

    ArXiv 도구(논문 검색 및 분석)와 Tavily 도구(웹 검색)를 함께 사용하여
    학술 논문과 최신 웹 정보를 모두 활용할 수 있는 에이전트를 생성합니다.

    Returns:
        CompiledStateGraph: 컴파일된 워크플로우 그래프
    """
    # ArXiv MCP 서버 구성
    server_configs = {
        "arxiv": {
            "command": os.path.join(
                ARXIV_MCP_PATH,
                ".venv",
                "Scripts" if sys.platform == "win32" else "bin",
                "python",
            ),
            "args": [os.path.join(ARXIV_MCP_PATH, "src", "server.py")],
            "transport": "stdio",
            "env": {
                "TRANSPORT": "stdio",
            },
        }
    }

    # MCP 클라이언트 생성 및 ArXiv 도구 로드
    client, mcp_tools = await setup_mcp_client(server_configs=server_configs)

    # Tavily 웹 검색 도구 추가
    tavily_tool = TavilySearch(max_results=3)
    all_tools = mcp_tools + [tavily_tool]

    print(f"\n총 {len(all_tools)}개 도구 사용:")
    for t in all_tools:
        print(f"  - {t.name}")

    # LLM 설정 및 도구 바인딩
    llm = init_chat_model("claude-sonnet-4-6", temperature=0)
    llm_with_tools = llm.bind_tools(all_tools)

    # 워크플로우 그래프 생성
    workflow = StateGraph(AgentState)

    async def agent_node(state: AgentState):
        """에이전트 노드: LLM을 호출하여 응답을 생성합니다"""
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    # ToolNode 생성: 도구 호출을 처리합니다
    tool_node = ToolNode(all_tools)

    # 그래프에 노드 추가
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", tool_node)

    # 엣지 정의
    workflow.add_edge(START, "agent")
    # 조건부 엣지: 도구 호출 필요 시 tools로, 아니면 종료
    workflow.add_conditional_edges("agent", tools_condition)
    # 도구 실행 후 다시 에이전트로
    workflow.add_edge("tools", "agent")

    # 그래프 컴파일
    app = workflow.compile(checkpointer=InMemorySaver())

    return app

In [ ]:
# 커스텀 워크플로우 생성
workflow_app = await create_arxiv_workflow()

In [ ]:
from IPython.display import Image, display

# 워크플로우 그래프 시각화
display(Image(workflow_app.get_graph().draw_mermaid_png()))

### ArXiv + Tavily 복합 검색

ArXiv로 학술 논문을 검색하고, Tavily로 최신 관련 뉴스나 블로그 포스트를 함께 조회합니다.

In [ ]:
# 새로운 대화 스레드 설정
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 복합 검색: 학술 논문 + 웹 정보
response = await astream_graph(
    workflow_app,
    inputs={
        "messages": [
            (
                "human",
                "AI Agent 관련 최신 arXiv 논문을 검색하고, "
                "웹에서 AI Agent 관련 최신 동향도 함께 검색해서 종합적으로 정리해주세요.",
            )
        ]
    },
    config=config,
)

---

## Part 8: 실전 활용 예제

### 예제 1: 연구 주제별 논문 서베이

특정 연구 주제에 대해 논문을 검색하고 체계적으로 분류하여 서베이 형태로 정리합니다.

In [ ]:
# 새로운 에이전트 생성 (깨끗한 상태에서 시작)
agent = await create_arxiv_agent()

config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 실전 예제 1: 연구 서베이
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "Prompt Engineering 기법에 대한 최신 논문을 5편 검색하고, "
                "각 논문의 핵심 기법을 표로 비교 정리해주세요. "
                "Chain-of-Thought, Few-shot, Zero-shot 등 다양한 기법을 포함해주세요.",
            )
        ]
    },
    config=config,
)

### 예제 2: 특정 논문의 심층 분석

유명 논문의 상세 정보를 가져와 방법론과 실험 결과를 심층 분석합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 실전 예제 2: 논문 심층 분석
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "arXiv 논문 ID '2302.13971'(LLaMA 논문)의 상세 정보를 조회하고, "
                "이 논문의 모델 아키텍처, 학습 데이터셋, 주요 실험 결과를 분석해주세요.",
            )
        ]
    },
    config=config,
)

### 예제 3: 연구 트렌드 분석

특정 분야의 최신 논문들을 검색하고 연구 트렌드를 분석합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 실전 예제 3: 연구 트렌드 분석
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "최신 코드 생성(Code Generation) 관련 arXiv 논문을 검색하고, "
                "현재 연구 트렌드를 분석해주세요. "
                "어떤 접근법이 주목받고 있는지, 향후 연구 방향은 어떻게 될지 예측해주세요.",
            )
        ]
    },
    config=config,
)

### 예제 4: 논문 간 비교 분석 (다중 턴)

여러 논문을 순차적으로 조회하고 비교 분석합니다. 동일한 `thread_id`를 유지하여 이전 조회 결과를 참조합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 첫 번째 질문: GPT-4 관련 논문 검색
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "GPT-4와 관련된 arXiv 논문을 검색해주세요. "
                "특히 GPT-4의 성능 평가나 분석을 다룬 논문을 찾아주세요.",
            )
        ]
    },
    config=config,
)

In [ ]:
# 두 번째 질문: 이전 결과 기반 비교 분석
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "위에서 찾은 논문들과 오픈소스 LLM(LLaMA, Mistral 등) 관련 논문을 비교하여, "
                "폐쇄형 모델과 오픈소스 모델의 성능 차이에 대한 분석을 해주세요.",
            )
        ]
    },
    config=config,
)

---

## 요약

이 튜토리얼에서는 ArXiv MCP 서버를 LangGraph와 통합하여 학술 논문을 검색하고 분석하는 전체 과정을 학습했습니다.

### 핵심 정리

| 개념 | 내용 |
|------|------|
| **ArXiv MCP** | arXiv 학술 논문 데이터베이스에 MCP 프로토콜로 접근하는 서버 |
| **search_arxiv** | 키워드 기반 arXiv 논문 검색 도구 |
| **get_paper_details** | 논문 ID로 상세 정보 및 PDF 내용 조회 도구 |
| **search_and_summarize** | 논문 검색 + 종합 요약을 한 번에 수행하는 도구 |
| **MCP 연결** | `stdio` 전송 방식 + `TRANSPORT=stdio` 환경 변수 설정 |
| **에이전트 통합** | `create_agent`로 자동 도구 호출 루프 구현 |
| **커스텀 워크플로우** | `ToolNode` + `StateGraph`로 세밀한 흐름 제어 |

### ArXiv MCP 활용 팁

1. **구체적인 검색 키워드 사용**: "AI" 보다는 "retrieval augmented generation for question answering"처럼 구체적일수록 정확한 결과 반환
2. **논문 ID 활용**: 알고 있는 논문의 arXiv ID로 직접 조회하면 정확한 정보 획득
3. **include_content 주의**: PDF 전문 추출은 시간이 걸릴 수 있으므로 필요한 경우에만 사용
4. **search_and_summarize 활용**: 빠른 서베이가 필요할 때 검색+요약을 한 번에 수행
5. **Tavily와 조합**: ArXiv(학술 논문) + Tavily(웹 검색) 조합으로 학술 정보와 실무 정보를 동시에 획득
6. **다중 턴 활용**: 검색 → 상세 조회 → 비교 분석 순서로 동일 thread_id에서 대화를 이어가면 효과적

### 다음 단계

- `06-MCP/` 폴더의 다른 MCP 튜토리얼을 통해 다양한 MCP 서버 활용법 학습
- [Smithery AI](https://smithery.ai/)에서 다양한 3rd Party MCP 서버 탐색
- ArXiv MCP + Context7 + Tavily를 결합한 종합 연구 보조 에이전트 구축